# Robotic-Arm Checkers Dataset Generator

Pipeline on a Kaggle notebook with **2 × T4 GPUs (16 GB each)** or a **Colab A100 (40 GB) or G4 (96 GB)**:

1. **Host Gemma-4-E4B-it on vLLM** with multimodal / image input enabled.
2. **Spawn a MuJoCo checkers scene** (adapted from `checkers_simulation.ipynb`) with the Trossen dual-arm robot.
3. **Load the three pretrained RL checkers models** from `checkers_games/`.
4. The **LLM plays one side**. Each turn it receives *only* a top-down camera image of the board, produces structured `<thinking>` + `<actions>` output, and those actions drive the left robot arm.
5. The **RL stack plays the other side**, selected each turn by a weighted policy:
   `{random-move, model1-linear, model2-medium, model3-deep}` where the weights are annealed so *random* dominates early and the deeper RL models dominate late-game.
6. Every turn from the LLM side is recorded as `(system, user_image, thinking, actions)`. After a full game we slice it into **windows of 1-6 consecutive turns** and emit each window as a SFT training sample.
7. The dataset is saved as **JSONL + Parquet + CSV index + PNG images**, ready for `datasets.load_dataset(...)`, HuggingFace Hub upload, or Kaggle Datasets upload.

> **Platform guide:**
>
> | Platform | Model | Strategy | Notes |
> |---|---|---|---|
> | **Kaggle 2×T4** | Gemma-4-E4B-it | fp16, TP=2 | Attach `google/gemma-4` + `checkers-games` datasets |
> | **Colab A100** | Gemma-4-E4B-it | fp16, TP=1 | Add `HF_TOKEN` to Colab Secrets; upload checkers-games zip via Files panel |
>
> **Colab setup (one-time):**
> 1. Click the **key icon** in the left sidebar → Secrets → add `HF_TOKEN` (your HuggingFace access token).
>    Make sure you've accepted the Gemma-4 licence at `huggingface.co/google/gemma-4-e4b-it`.
> 2. In the **Files panel** (folder icon), upload your `checkers-games.zip` to `/content/`.
>    The notebook will detect and unzip it automatically.

## 0. CUDA upgrade

On **Kaggle** and **Colab**, the default CUDA is 12.8 (at the time of creating this notebook) but vLLM 0.20.0 was built against 13.0, so we upgrade PyTorch and restart.

In [ ]:
import torch
print(torch.version.cuda)

12.8


In [ ]:
import os, subprocess, torch

# system CUDA is 12.8; upgrade toolkit to 13.0 then pull matching PyTorch wheels.
subprocess.run('wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb', shell=True)
subprocess.run('sudo dpkg -i cuda-keyring_1.1-1_all.deb 2>/dev/null', shell=True)
subprocess.run('sudo apt-get -qq update && sudo apt-get -y install cuda-toolkit-13-0', shell=True)
subprocess.run('pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130', shell=True)
os.environ['PATH']            = os.environ.get('PATH', '')            + ':/usr/local/cuda-13.1/bin'
os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':/usr/local/cuda-13.1/lib64'
print('CUDA 13.1 toolkit + PyTorch cu131 installed.')

CUDA 13.1 toolkit + PyTorch cu131 installed.


In [ ]:
# Automatically restart the session to clear the old torch 12.8 from memory
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
print(torch.version.cuda)

13.0


## 1. Install dependencies

In [ ]:
import os, sys, subprocess

#  Platform detection (used in all subsequent cells) 
try:
    import google.colab as _gc  # noqa: F401
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = os.path.exists('/content') and not os.path.exists('/kaggle/working')
_IS_KAGGLE = os.path.exists('/kaggle/working')

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

pip('--upgrade', 'pip')

import torch as _t
_cv = _t.version.cuda.split('.')          # read actual CUDA version, not hardcoded
_cu_tag = 'cu' + _cv[0] + _cv[1]         # e.g. 'cu130' on Kaggle, 'cu121' on Colab A100
pip('vllm==0.20.0',
    '--extra-index-url', 'https://download.pytorch.org/whl/' + _cu_tag)

if _IS_KAGGLE:
    # Pin back packages that vLLM's resolver upgrades beyond what Kaggle's environment
    # (tensorflow, google-adk, RAPIDS) accepts.
    pip('protobuf>=5.26.1,<6.0.0dev',
        'opentelemetry-api>=1.36.0,<1.40.0',
        'opentelemetry-sdk>=1.36.0,<1.40.0',
        'numba<0.62.0',
    )

pip('openai>=1.40.0', 'transformers>=4.48.0', 'accelerate>=0.34.0')
pip('huggingface_hub>=0.23.0')
pip('mujoco>=3.2.0', 'dm_control', 'opencv-python', 'imageio[ffmpeg]')
pip('datasets>=2.21.0', 'pyarrow>=14.0.0', 'pillow', 'polars')

subprocess.run('apt-get -qq update 2>/dev/null || true', shell=True)
subprocess.run('apt-get -qq install --fix-missing -y libgl1 libosmesa6 libglew2.2 ffmpeg',
               shell=True, check=True)

os.environ['MUJOCO_GL']              = 'osmesa'
os.environ['PYOPENGL_PLATFORM']      = 'osmesa'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['VLLM_TRITON_UNIFIED_ATTN_BLOCK_SIZE'] = '32'

print(f'Platform: {"Colab" if _IS_COLAB else "Kaggle" if _IS_KAGGLE else "other"}')
print(f'CUDA: {_t.version.cuda}  →  index tag: {_cu_tag}')

Platform: Colab
CUDA: 13.0  →  index tag: cu130


In [ ]:
import zipfile
from pathlib import Path

REPO_DIR = Path.cwd() / 'trossen_arm_mujoco'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/TrossenRobotics/trossen_arm_mujoco.git',
                    str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

# Search for checkers_core.py. Order: Kaggle input → Colab /content → cwd.
CHECKERS_DIR = None
search_roots = [
    Path('/kaggle/input'),
    Path('/content')
]
for root in search_roots:
    if not root.exists():
        continue
    for match in root.rglob('checkers_core.py'):
        CHECKERS_DIR = match.parent
        break
    if CHECKERS_DIR:
        break

# Colab: if not found yet, scan /content/ for any uploaded zip that contains checkers_core.py
# and extract it automatically. Upload the zip via the Files panel (folder icon) first.
if CHECKERS_DIR is None and _IS_COLAB:
    for _z in sorted(Path('/content').glob('*.zip')):
        try:
            with zipfile.ZipFile(_z) as _zf:
                if any('checkers_core.py' in n for n in _zf.namelist()):
                    _out = Path('/content') / _z.stem
                    print(f'Extracting {_z.name} → {_out}')
                    _zf.extractall(_out)
                    for match in _out.rglob('checkers_core.py'):
                        CHECKERS_DIR = match.parent
                        break
                    break
        except zipfile.BadZipFile:
            continue

if CHECKERS_DIR is None:
    raise FileNotFoundError(
        'Could not locate checkers_core.py.\n'
        '  Kaggle: attach the akhilvardhan/checkers-games dataset as a notebook input.\n'
        '  Colab:  upload your checkers-games zip to /content/ via the Files panel (folder icon).'
    )

sys.path.insert(0, str(CHECKERS_DIR))
print('checkers_games @', CHECKERS_DIR)

Extracting checkers_games.zip → /content/checkers_games
checkers_games @ /content/checkers_games/checkers_games


## 2. Configuration

In [ ]:
import torch as _torch_cfg

# Auto-detect GPU count to set tensor-parallel width:
#   Kaggle 2×T4  → 2 GPUs → TP=2
#   Colab A100   → 1 GPU  → TP=1  (no TP overhead; full 40/80 GB for weights + KV cache)
_GPU_COUNT = _torch_cfg.cuda.device_count()

class CFG:
    #  model selection 
    # Gemma-4-E4B-it works on both platforms:
    #   Kaggle 2×T4  : loaded from the attached dataset at /kaggle/input/..., fp16, TP=2
    #   Colab  A100  : pulled from HuggingFace Hub via vLLM, fp16, TP=1
    #
    # To use Gemma-4-31B-it on an 80 GB A100:
    #   model_path_primary = 'google/gemma-4-31b-it'   (needs HF_TOKEN secret)
    #   quantization       = 'awq'                     (use a pre-quantised AWQ checkpoint)
    #   tensor_parallel    = 1                         (manual override)
    #
    use_fallback        = False   # E4B is primary on both platforms
    model_path_primary  = (
        '/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1'
        if _IS_KAGGLE else
        'google/gemma-4-e4b-it'   # vLLM downloads from HF Hub (needs HF_TOKEN in Colab Secrets)
    )
    model_path_fallback = model_path_primary   # same model; fallback path kept for compatibility
    served_name         = 'gemma-vla'
    port                = 8000
    context_tokens      = 8192
    max_new_tokens      = 512
    temperature         = 0.7
    top_p               = 0.9
    gpu_memory_util     = 0.90
    # TP=2 on Kaggle (2×T4, 16 GB each); TP=1 on Colab (single A100, 40/80 GB)
    tensor_parallel     = _GPU_COUNT if _IS_KAGGLE else 1
    server_timeout_s    = 600
    request_timeout_s   = 240

    # quantization
    # E4B fp16 (~8 GB) fits on both platforms without any quantization.
    quantization   = None
    kv_cache_dtype = 'auto'   # TRITON_ATTN compatible
    num_games           = 5
    max_plies           = 60
    window_min          = 1
    window_max          = 6
    windows_per_game    = 6
    seed                = 42

    #  Best-of-N sampling / validation 
    candidates_per_turn  = 4
    sampling_temperature = 0.9
    max_retries          = 2
    min_score_accept     = 6.0
    hard_gate_structure  = True
    hard_gate_parseable  = True
    score_weights = {
        'structure':   1.0,
        'parseable':   1.0,
        'op_sequence': 1.0,
        'from_to':     1.0,
        'legal':       3.0,
        'captures':    1.0,
        'heuristic':   0.5,
    }

    opponent_weights_early = {'random': 0.45, 'model1': 0.25, 'model2': 0.20, 'model3': 0.10}
    opponent_weights_late  = {'random': 0.05, 'model1': 0.15, 'model2': 0.30, 'model3': 0.50}

    output_dir = Path(
        '/kaggle/working/robot_checkers_dataset' if _IS_KAGGLE else '/content/robot_checkers_dataset'
    )
    images_dir = output_dir / 'images'
    videos_dir = output_dir / 'videos'

CFG.output_dir.mkdir(parents=True, exist_ok=True)
CFG.images_dir.mkdir(parents=True, exist_ok=True)
CFG.videos_dir.mkdir(parents=True, exist_ok=True)
print('Platform       ->', 'Kaggle' if _IS_KAGGLE else 'Colab' if _IS_COLAB else 'other')
print('Model          ->', CFG.model_path_primary)
print('Tensor parallel->', CFG.tensor_parallel, f'(GPUs detected: {_GPU_COUNT})')
print('Quantization   ->', CFG.quantization or 'none (fp16)')
print('KV cache dtype ->', CFG.kv_cache_dtype)
print('Context tokens ->', CFG.context_tokens)
print('Output dir     ->', CFG.output_dir)

Platform       -> Colab
Model          -> google/gemma-4-e4b-it
Tensor parallel-> 1 (GPUs detected: 1)
Quantization   -> none (fp16)
KV cache dtype -> auto
Context tokens -> 8192
Output dir     -> /content/robot_checkers_dataset


In [ ]:
!vllm --version

0.20.0


## 3. Launch the vLLM server

In [ ]:
import time
from openai import OpenAI

# Kill any leftover vLLM processes from a previous run
subprocess.run(['pkill', '-9', '-f', 'vllm.entrypoints.openai.api_server'],
               capture_output=True)
time.sleep(5)

# Kaggle T4 pairs have no NVLink — NCCL P2P/SHM must be disabled to avoid hangs.
# Colab A100 has NVLink; keep P2P/SHM enabled for better inter-GPU bandwidth.
if _IS_KAGGLE:
    os.environ['NCCL_P2P_DISABLE'] = '1'
    os.environ['NCCL_SHM_DISABLE'] = '1'

# On Colab: inject HuggingFace token so vLLM can pull the gated Gemma-4 model from the Hub.
if _IS_COLAB and 'HUGGING_FACE_HUB_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HUGGING_FACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')
        print('HF token loaded from Colab Secrets.')
    except Exception:
        print('Warning: HF_TOKEN not found in Colab Secrets. '
              'Add it via the key icon in the sidebar if the model download fails.')

MODEL_PATH = CFG.model_path_primary

# For local paths (Kaggle), verify the directory exists before starting the server.
# HF Hub model IDs (e.g. 'google/gemma-4-e4b-it') contain a '/' but no leading '/'.
_is_local_path = MODEL_PATH.startswith('/')
if _is_local_path:
    assert Path(MODEL_PATH).exists(), (
        f'Model not found at {MODEL_PATH}.\n'
        f'  Kaggle: attach the google/gemma-4 dataset as a notebook input.\n'
        f'  Colab:  the model_path_primary should be an HF model ID like "google/gemma-4-e4b-it".'
    )

# Memory budget:
#   Kaggle 2×T4 (32 GB total):  E4B fp16 ~8 GB weights + fp8 KV cache ~3-5 GB → fits with TP=2
#   Colab A100  (40 GB):        E4B fp16 ~8 GB weights + fp8 KV cache ~5 GB   → fits with headroom
#
# --max-num-batched-tokens 4096: Gemma-4 bidirectional MM attention forces --disable_chunked_mm_input;
#   image encoder emits 2496 tokens/image, so this must be >= 2496.
# --enforce-eager: disables CUDA-graph capture; required for Gemma-4 heterogeneous-head-dim in TP mode.

quant_flags = (
    ['--quantization', CFG.quantization]
    if CFG.quantization is not None else []
)

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_PATH,
    '--served-model-name', CFG.served_name,
    '--host', '0.0.0.0', '--port', str(CFG.port),
    '--tensor-parallel-size', str(CFG.tensor_parallel),
    '--gpu-memory-utilization', str(CFG.gpu_memory_util),
    '--max-model-len', str(CFG.context_tokens),
    '--max-num-batched-tokens', '4096',
    '--dtype', 'float16',
    '--kv-cache-dtype', CFG.kv_cache_dtype,
    '--limit-mm-per-prompt', '{"image": 4}',
    '--max-num-seqs', str(CFG.candidates_per_turn * 2),
    '--enforce-eager',
    '--enable-prefix-caching',
    *quant_flags,
]

log_path = CFG.output_dir / 'vllm_server.log'
log_fh   = open(log_path, 'w')
server   = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT, start_new_session=True)
print('vLLM PID:', server.pid, '| logs:', log_path)
print('Model:', MODEL_PATH)
print('Tensor parallel:', CFG.tensor_parallel)
print('Quantization:', CFG.quantization or 'none (fp16)')
print('KV cache dtype:', CFG.kv_cache_dtype)
print('Max model len:', CFG.context_tokens)

client = OpenAI(base_url=f'http://127.0.0.1:{CFG.port}/v1', api_key='sk-local', timeout=CFG.request_timeout_s)
start  = time.time()
while time.time() - start < CFG.server_timeout_s:
    if server.poll() is not None:
        raise RuntimeError(f'vLLM died ({server.returncode}). Inspect {log_path}.')
    try:
        client.models.list(); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('vLLM did not start in time.')
print(f'vLLM ready in {time.time()-start:.1f}s')

vLLM PID: 6534 | logs: /content/robot_checkers_dataset/vllm_server.log
Model: google/gemma-4-e4b-it
Tensor parallel: 1
Quantization: none (fp16)
KV cache dtype: auto
Max model len: 8192
vLLM ready in 158.6s


## 4. MuJoCo checkers scene

Adapted from `checkers_simulation.ipynb` with **red pieces placed on rows 5-7 (bottom)** so the axis matches `checkers_core.initializeBoard()` where `'R'` is at the bottom.

In [ ]:
import xml.etree.ElementTree as ET
import numpy as np, mujoco, cv2

_src_dir = str(REPO_DIR / 'trossen_arm_mujoco' / 'src')
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

from controller import Controller, RobotType

ASSETS = REPO_DIR / 'trossen_arm_mujoco' / 'assets' / 'stationary_ai'
BASE   = ASSETS / 'scene.xml'
GEN    = ASSETS / 'scene_checkers_rl.xml'

BOARD_CENTER, SQUARE_SIZE, BOARD_TOP_Z = np.array([0.02, 0.0]), 0.07, 0.0205
SQUARE_HALF = np.array([SQUARE_SIZE/2, SQUARE_SIZE/2, 0.002])
PIECE_R, PIECE_HH = 0.022, 0.008
PIECE_Z = BOARD_TOP_Z + SQUARE_HALF[2] + PIECE_HH

def square_to_world(row, col):
    x = BOARD_CENTER[0] + (col - 3.5) * SQUARE_SIZE
    y = BOARD_CENTER[1] + (3.5 - row) * SQUARE_SIZE
    return np.array([x, y, PIECE_Z])

def square_name(row, col):
    return f"{chr(ord('a')+col)}{8-row}"

def name_to_rc(name):
    col = ord(name[0]) - ord('a')
    row = 8 - int(name[1])
    return row, col

def build_scene():
    tree = ET.parse(BASE); root = tree.getroot()
    asset, world = root.find('asset'), root.find('worldbody')
    vis = root.find('visual')
    if vis is None:
        vis = ET.SubElement(root, 'visual')
    g = vis.find('global')
    if g is None:
        g = ET.SubElement(vis, 'global')
    g.set('offwidth', '1280'); g.set('offheight', '1280')

    for n, rgba in [('checker_light','0.90 0.82 0.67 1'), ('checker_dark','0.27 0.17 0.11 1'),
                    ('piece_red','0.75 0.12 0.12 1'),     ('piece_black','0.12 0.12 0.12 1')]:
        ET.SubElement(asset, 'material', name=n, rgba=rgba)

    board = ET.SubElement(world, 'body', name='checkers_board', pos='0 0 0')
    ET.SubElement(board, 'geom', name='board_base', type='box',
                  pos=f'{BOARD_CENTER[0]} {BOARD_CENTER[1]} {BOARD_TOP_Z}',
                  size=f'{SQUARE_SIZE*4:.4f} {SQUARE_SIZE*4:.4f} 0.002',
                  material='checker_dark', contype='0', conaffinity='0')

    ET.SubElement(world, 'camera', name='cam_board_top',
                  pos=f'{BOARD_CENTER[0]:.4f} {BOARD_CENTER[1]:.4f} 0.95',
                  quat='1 0 0 0', mode='fixed', fovy='40')
    ET.SubElement(world, 'camera', name='cam_board_oblique',
                  pos=f'{BOARD_CENTER[0]+0.35:.4f} {BOARD_CENTER[1]:.4f} 0.70',
                  xyaxes='0 1 0 -0.6 0 0.8', mode='fixed', fovy='42')

    for r in range(8):
        for c in range(8):
            x,y,_ = square_to_world(r,c)
            mat = 'checker_dark' if (r+c)%2 else 'checker_light'
            ET.SubElement(board, 'geom', name=f'sq_{r}_{c}', type='box',
                          pos=f'{x:.4f} {y:.4f} {BOARD_TOP_Z+SQUARE_HALF[2]:.4f}',
                          size=f'{SQUARE_HALF[0]:.4f} {SQUARE_HALF[1]:.4f} {SQUARE_HALF[2]:.4f}',
                          material=mat, contype='0', conaffinity='0')

    pieces=[]; state={}
    for color, rows in [('red',[5,6,7]),('black',[0,1,2])]:
        idx = 0
        for r in rows:
            for c in range(8):
                if (r+c)%2==1:
                    bn  = f'{color}_piece_{idx}'; jn=f'{bn}_joint'; gn=f'{bn}_geom'
                    pos = square_to_world(r,c); sq = square_name(r,c)
                    b = ET.SubElement(world, 'body', name=bn, pos=f'{pos[0]:.4f} {pos[1]:.4f} {pos[2]:.4f}')
                    ET.SubElement(b, 'joint', name=jn, type='free')
                    ET.SubElement(b, 'geom', name=gn, type='cylinder',
                                  size=f'{PIECE_R:.4f} {PIECE_HH:.4f}',
                                  material='piece_red' if color=='red' else 'piece_black', mass='0.01')
                    ET.SubElement(b, 'site', name=f'{bn}_site', size='0.002')
                    pieces.append(dict(name=bn, joint=jn, color=color, square=sq, row=r, col=c))
                    state[sq] = bn
                    idx += 1
    ET.indent(tree, space='  '); tree.write(GEN, encoding='utf-8', xml_declaration=True)
    return pieces, state

pieces, board_state = build_scene()
print('Scene ->', GEN, '| pieces:', len(pieces))

Scene -> /content/trossen_arm_mujoco/trossen_arm_mujoco/assets/stationary_ai/scene_checkers_rl.xml | pieces: 24


In [ ]:
model = mujoco.MjModel.from_xml_path(str(GEN))
data  = mujoco.MjData(model)

L_ARM = [f'follower_left_joint_{i}' for i in range(6)]
R_ARM = [f'follower_right_joint_{i}' for i in range(6)]
L_GR  = ['follower_left_left_carriage_joint']
R_GR  = ['follower_right_left_carriage_joint']

left_ctl  = Controller(model=model, data=data, robot_type=RobotType.STATIONARY_AI,
                       ee_site_name='follower_left_ee_site', arm_joint_names=L_ARM,
                       gripper_joint_names=L_GR, ik_scale=1.0, ik_damping=0.03)
right_ctl = Controller(model=model, data=data, robot_type=RobotType.STATIONARY_AI,
                       ee_site_name='follower_right_ee_site', arm_joint_names=R_ARM,
                       gripper_joint_names=R_GR, ik_scale=1.0, ik_damping=0.03)

L_QUAT = np.array([0.5, 0.5, 0.5, -0.5])
R_QUAT = np.array([0.5,-0.5, 0.5,  0.5])
L_HOME = np.array([0.05,  0.28, 0.28])
R_HOME = np.array([0.05, -0.28, 0.28])
APPROACH_Z, PICK_Z, CARRY_Z = 0.18, 0.065, 0.20
CARRY_OFFSET = np.array([0.0, 0.0, -0.03])
GR_OPEN, GR_CLOSED = 0.044, 0.022

piece_by_name = {p['name']: p for p in pieces}
CAMERAS = ['cam_board_top', 'cam_board_oblique']
renderers = {c: mujoco.Renderer(model, 1024, 1024) for c in CAMERAS}

def set_piece(name, pos, quat=None):
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, piece_by_name[name]['joint'])
    qa  = model.jnt_qposadr[jid]; va = model.jnt_dofadr[jid]
    data.qpos[qa:qa+3] = pos
    data.qpos[qa+3:qa+7] = np.array([1,0,0,0.0]) if quat is None else quat
    data.qvel[va:va+6]   = 0

def reset_robots():
    for jn in L_ARM + R_ARM:
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jn)
        data.qpos[model.jnt_qposadr[jid]] = 0.0
    left_ctl.open_gripper(); right_ctl.open_gripper()
    mujoco.mj_forward(model, data)

def reset_scene():
    reset_robots()
    board_state.clear()
    for p in pieces:
        p['square'] = square_name(p['row'], p['col'])
        board_state[p['square']] = p['name']
        set_piece(p['name'], square_to_world(p['row'], p['col']))
    mujoco.mj_forward(model, data)

def render(camera='cam_board_top'):
    r = renderers[camera]; r.update_scene(data, camera=camera)
    return r.render().copy()

def step_sim(n=1):
    for _ in range(n): mujoco.mj_step(model, data)

def move_arm(ctl, tgt_pos, tgt_quat, gr, attached=None, iters=120):
    for _ in range(iters):
        ctl.set_gripper_position(gr)
        err = ctl.set_ee_pose(tgt_pos, tgt_quat, position_only=False)
        step_sim(1)
        if attached is not None:
            ee,_ = ctl.get_ee_pose(); set_piece(attached, ee + CARRY_OFFSET); mujoco.mj_forward(model, data)
        if err < 0.015: break
    step_sim(10)

def arm_for(color):
    return (left_ctl, L_HOME, L_QUAT) if color == 'red' else (right_ctl, R_HOME, R_QUAT)

def world_of(sq): r,c = name_to_rc(sq); return square_to_world(r,c)

def execute_robot_move(from_sq, to_sq, color, captured_sqs=()):
    piece = board_state.get(from_sq)
    assert piece is not None, f'No piece on {from_sq}'
    ctl, home, quat = arm_for(color)

    for csq in captured_sqs:
        cname = board_state.get(csq)
        if cname is None: continue
        move_arm(ctl, world_of(csq)+np.array([0,0,APPROACH_Z]), quat, GR_OPEN)
        move_arm(ctl, world_of(csq)+np.array([0,0,PICK_Z]),     quat, GR_OPEN)
        move_arm(ctl, world_of(csq)+np.array([0,0,PICK_Z]),     quat, GR_CLOSED)
        move_arm(ctl, home + np.array([0, 0.05 if color=='red' else -0.05, 0.05]), quat, GR_CLOSED, attached=cname)
        set_piece(cname, np.array([0.7, 0.4 if color=='red' else -0.4, -0.2]))
        move_arm(ctl, home, quat, GR_OPEN)
        board_state[csq] = None

    src, tgt = world_of(from_sq), world_of(to_sq)
    move_arm(ctl, src + np.array([0,0,APPROACH_Z]), quat, GR_OPEN)
    move_arm(ctl, src + np.array([0,0,PICK_Z]),     quat, GR_OPEN)
    move_arm(ctl, src + np.array([0,0,PICK_Z]),     quat, GR_CLOSED)
    move_arm(ctl, src + np.array([0,0,APPROACH_Z+0.02]), quat, GR_CLOSED, attached=piece)
    move_arm(ctl, tgt + np.array([0,0,CARRY_Z]),    quat, GR_CLOSED, attached=piece)
    move_arm(ctl, tgt + np.array([0,0,PICK_Z]),     quat, GR_CLOSED, attached=piece)
    set_piece(piece, tgt); mujoco.mj_forward(model, data)
    move_arm(ctl, tgt + np.array([0,0,PICK_Z]),     quat, GR_OPEN)
    move_arm(ctl, home, quat, GR_OPEN)

    board_state[from_sq] = None
    board_state[to_sq]   = piece
    piece_by_name[piece]['square'] = to_sq

reset_scene()
print('Scene ready. Initial top-down frame shape:', render().shape)

Scene ready. Initial top-down frame shape: (1024, 1024, 3)


## 5. RL bridge (checkers_core <-> MuJoCo)

`checkers_core.possibleMoves` returns full next-boards, so we diff two boards to recover `(from, to, captured)`.

In [ ]:
import copy, random, torch
from checkers_core import (
    LinearModel, MediumModel, DeepModel,
    initializeBoard, possibleMoves, chooseMove, chooseRandomMove,
    isGameOver, getFinalBoardScore,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def load_rl(cls, fname):
    m = cls().to(device)
    m.load_state_dict(torch.load(CHECKERS_DIR / fname, map_location=device, weights_only=True))
    m.eval(); return m

rl_models = {
    'model1': load_rl(LinearModel, 'model1_weights.pt'),
    'model2': load_rl(MediumModel, 'model2_weights.pt'),
    'model3': load_rl(DeepModel,   'model3_weights.pt'),
}
print('RL models loaded:', list(rl_models))

def diff_boards(before, after):
    froms, tos = [], []
    for r in range(8):
        for c in range(8):
            b, a = before[r][c], after[r][c]
            if b != a:
                if b != ' ' and a == ' ':     froms.append((r,c,b))
                elif b == ' ' and a != ' ':   tos.append((r,c,a))
    from_sq = to_sq = None
    for (fr,fc,fp) in froms:
        for (tr,tc,tp) in tos:
            if fp in (tp, tp.replace('K','')) or tp in (fp, fp.replace('K','')):
                from_sq = square_name(fr, fc); to_sq = square_name(tr, tc); break
        if from_sq: break
    mover_rc = name_to_rc(from_sq) if from_sq else None
    captured = [square_name(fr, fc) for (fr,fc,_) in froms if (fr,fc) != mover_rc]
    return from_sq, to_sq, tuple(captured)

def select_opponent_policy(ply_idx, max_plies):
    t = min(1.0, ply_idx / max(1, max_plies))
    w = {k: (1-t)*CFG.opponent_weights_early[k] + t*CFG.opponent_weights_late[k]
         for k in CFG.opponent_weights_early}
    keys, vals = zip(*w.items())
    return random.choices(keys, weights=vals, k=1)[0], w

def rl_pick_move(core_board, pin, opponent_pin, policy):
    if policy == 'random':
        return chooseRandomMove(core_board, pin)
    return chooseMove(rl_models[policy], core_board, pin, opponent_pin, device)

RL models loaded: ['model1', 'model2', 'model3']


## 6. LLM side: system prompt, parsing, validation, best-of-N sampling

- `parse_llm_output` splits the `<thinking>` / `<actions>` blocks and JSON-parses the actions.
- `validate_candidate` runs 8 independent checks (structure, labels, parseable, allowed ops, skeleton ordering, (from,to) recoverable, legality against `checkers_core`, and captures-count match).
- `score_candidate` combines those booleans with a `model3`-based heuristic on the resulting board.
- `ask_llm_best_of_n` issues a single vLLM request with `n=CFG.candidates_per_turn`. If every candidate fails the hard gates (`<thinking>`/`<actions>` missing or actions not JSON), it retries up to `CFG.max_retries` more rounds with different seeds. Early-exit as soon as a candidate with a fully legal move crosses `CFG.min_score_accept`.

In [ ]:
import base64, io, json, re
from PIL import Image

ALLOWED_OPS = {
    'move_above', 'descend_to_pick', 'close_gripper', 'lift',
    'move_carry', 'descend_to_place', 'open_gripper', 'retreat_home',
    'remove_captured',
}
# Expected skeleton for a single-move plan (captures allowed as a leading block).
CORE_SEQUENCE = (
    'move_above', 'descend_to_pick', 'close_gripper', 'lift',
    'move_carry', 'descend_to_place', 'open_gripper', 'retreat_home',
)
SQUARE_RE = re.compile(r'^[a-h][1-8]$')

def system_prompt_for(camera_name):
    cam_desc = {
        'cam_board_top':     'a camera pointing straight down from 0.95 m above the centre of the board',
        'cam_board_oblique': 'a camera placed ~0.35 m to your right and 0.70 m above the board, angled ~35 degrees off vertical',
    }[camera_name]
    return (
        'You are the control policy for a two-arm Trossen WidowX robot playing checkers against an autonomous opponent.\n'
        f'Your only sensor is {cam_desc}. The full 8x8 board is always in frame.\n'
        'Board frame: files a..h run left-to-right in the image, ranks 1..8 run bottom-to-top. '
        'You control the RED pieces (bottom of the board). The left arm handles red.\n'
        'Each piece sits on its square. Only diagonal dark squares are legal. Men move toward the far side; '
        'kings move both ways.\n'
        '\n'
        'Robot primitives you may emit (and nothing else):\n'
        '  move_above(square)           -- approach pose above a square (z~0.18 m)\n'
        '  descend_to_pick(square)      -- lower to pick height (z~0.065 m)\n'
        '  close_gripper()              -- grasp the piece\n'
        '  lift(square)                 -- lift to carry height (z~0.20 m)\n'
        '  move_carry(square)           -- translate in carry pose above a target square\n'
        '  descend_to_place(square)     -- lower to place height\n'
        '  open_gripper()               -- release\n'
        '  retreat_home()               -- return to home pose\n'
        '  remove_captured(square)      -- pick the opponent piece on `square` and drop it in the capture tray\n'
        '\n'
        'OUTPUT FORMAT (STRICT, non-negotiable, validators enforce this):\n'
        '<thinking>\n'
        '  STATE: <concise snapshot of red/black pieces with files/ranks>\n'
        '  PLAN: <candidate moves considered, (from, to) picked, short justification>\n'
        '  ACTIONS: <1-2 line summary of the primitive sequence you will emit>\n'
        '</thinking>\n'
        '<actions>\n'
        '[{"op": "move_above",       "arg": "b6"},\n'
        ' {"op": "descend_to_pick",  "arg": "b6"},\n'
        ' {"op": "close_gripper",    "arg": null},\n'
        ' {"op": "lift",             "arg": "b6"},\n'
        ' {"op": "move_carry",       "arg": "a5"},\n'
        ' {"op": "descend_to_place", "arg": "a5"},\n'
        ' {"op": "open_gripper",     "arg": null},\n'
        ' {"op": "retreat_home",     "arg": null}]\n'
        '</actions>\n'
        '\n'
        'Rules:\n'
        '  - The <thinking> block MUST contain the three labels STATE:, PLAN:, ACTIONS:.\n'
        '  - The <actions> block MUST be a valid JSON list whose every element is an object '
        '{"op": <primitive-name>, "arg": <square-or-null>}.\n'
        '  - Every capture must use one remove_captured(<captured-square>) call BEFORE the core '
        'pick-and-place block, and the total number of remove_captured calls must equal the number of '
        'opponent pieces jumped by this move.\n'
        '  - The core pick-and-place block MUST follow the exact order move_above -> descend_to_pick -> '
        'close_gripper -> lift -> move_carry -> descend_to_place -> open_gripper -> retreat_home.\n'
        '  - All square args use the form <file><rank>, e.g. b6.\n'
        '  - Emit exactly one checkers move per turn. Finish immediately after </actions>.\n'
    )

def image_to_data_url(arr):
    im = Image.fromarray(arr)
    buf = io.BytesIO(); im.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

THINK_RE   = re.compile(r'<thinking>(.*?)</thinking>', re.DOTALL | re.IGNORECASE)
ACTIONS_RE = re.compile(r'<actions>(.*?)</actions>',   re.DOTALL | re.IGNORECASE)

def parse_llm_output(text):
    t = THINK_RE.search(text);   a = ACTIONS_RE.search(text)
    thinking     = t.group(1).strip() if t else ''
    actions_raw  = a.group(1).strip() if a else ''
    parse_error  = None
    actions      = []
    if actions_raw:
        try:
            parsed = json.loads(actions_raw)
            if isinstance(parsed, list):
                actions = parsed
            else:
                parse_error = 'actions JSON is not a list'
        except Exception as exc:
            parse_error = f'actions JSON parse error: {exc}'
    else:
        parse_error = 'missing <actions> block'
    return {
        'has_thinking': t is not None,
        'has_actions':  a is not None,
        'thinking':     thinking,
        'actions':      actions,
        'actions_raw':  actions_raw,
        'parse_error':  parse_error,
    }

def extract_from_to(actions):
    from_sq = to_sq = None
    for step in actions:
        op  = step.get('op') if isinstance(step, dict) else None
        arg = step.get('arg') if isinstance(step, dict) else None
        if op == 'descend_to_pick' and from_sq is None: from_sq = arg
        elif op == 'descend_to_place':                   to_sq   = arg
    return from_sq, to_sq

#  validation 

def validate_candidate(parsed, legal_moves):
    """Return a dict of booleans / sub-scores describing how well the candidate matches spec.
    legal_moves: dict {(from, to): (next_board, captured_sqs)} from the current board for pin R.
    """
    issues = []
    # (1) structure: <thinking> / <actions> present.
    has_structure = parsed['has_thinking'] and parsed['has_actions']
    if not has_structure: issues.append('missing thinking/actions tags')

    # (2) thinking contains STATE: / PLAN: / ACTIONS: labels.
    thinking = parsed['thinking'].upper()
    has_labels = all(tok in thinking for tok in ('STATE:', 'PLAN:', 'ACTIONS:'))
    if has_structure and not has_labels: issues.append('thinking missing STATE/PLAN/ACTIONS labels')

    # (3) actions parseable.
    parseable = parsed['parse_error'] is None and isinstance(parsed['actions'], list) and len(parsed['actions']) > 0
    if not parseable: issues.append(parsed['parse_error'] or 'empty actions list')

    # (4) every action is a well-formed dict with an allowed op.
    ops_ok = parseable
    bad_entries = []
    if parseable:
        for i, step in enumerate(parsed['actions']):
            if not isinstance(step, dict) or 'op' not in step:
                ops_ok = False; bad_entries.append(i); continue
            if step['op'] not in ALLOWED_OPS:
                ops_ok = False; bad_entries.append(i)
            arg = step.get('arg')
            if arg is not None and not (isinstance(arg, str) and SQUARE_RE.match(arg)):
                if step['op'] not in ('close_gripper','open_gripper','retreat_home'):
                    ops_ok = False; bad_entries.append(i)
    if not ops_ok: issues.append(f'bad action entries @ {bad_entries}')

    # (5) skeleton: leading remove_captured*, then CORE_SEQUENCE in order.
    seq_ok = False
    captures_declared = 0
    if parseable:
        ops = [s.get('op') for s in parsed['actions'] if isinstance(s, dict)]
        i = 0
        while i < len(ops) and ops[i] == 'remove_captured':
            captures_declared += 1; i += 1
        core = ops[i:]
        seq_ok = tuple(core[:len(CORE_SEQUENCE)]) == CORE_SEQUENCE
    if not seq_ok: issues.append('primitive order does not match expected skeleton')

    # (6) recoverable (from, to).
    from_sq, to_sq = (None, None)
    if parseable:
        from_sq, to_sq = extract_from_to(parsed['actions'])
    from_to_ok = from_sq is not None and to_sq is not None \
                 and bool(SQUARE_RE.match(from_sq or '')) and bool(SQUARE_RE.match(to_sq or ''))
    if not from_to_ok: issues.append('could not parse (from, to) from actions')

    # (7) legality.
    is_legal = (from_sq, to_sq) in legal_moves
    if from_to_ok and not is_legal: issues.append(f'({from_sq}->{to_sq}) not in legal moves')

    # (8) captures match reality.
    captures_ok = False
    expected_caps = 0
    if is_legal:
        _, captured = legal_moves[(from_sq, to_sq)]
        expected_caps = len(captured)
        captures_ok = (captures_declared == expected_caps)
        if not captures_ok: issues.append(f'declared {captures_declared} captures, expected {expected_caps}')

    return {
        'has_structure':     has_structure,
        'has_labels':        has_labels,
        'parseable':         parseable,
        'ops_ok':            ops_ok,
        'seq_ok':            seq_ok,
        'from_to_ok':        from_to_ok,
        'is_legal':          is_legal,
        'captures_ok':       captures_ok,
        'from_sq':           from_sq,
        'to_sq':             to_sq,
        'captures_declared': captures_declared,
        'captures_expected': expected_caps,
        'issues':            issues,
    }

def heuristic_next_board_value(next_board):
    """RL model3 is the strongest - evaluate the board as seen from the LLM's (red) perspective."""
    from checkers_core import getFeatures
    feats = getFeatures(next_board, 'R', 'B')
    t = torch.FloatTensor(feats).unsqueeze(0).to(device)
    with torch.no_grad():
        v = rl_models['model3'](t).item()
    return max(-1.0, min(1.0, v / 100.0))

def score_candidate(v, legal_moves):
    w = CFG.score_weights
    s = 0.0
    s += w['structure']   * (1.0 if v['has_structure'] and v['has_labels'] else 0.0)
    s += w['parseable']   * (1.0 if v['parseable'] and v['ops_ok'] else 0.0)
    s += w['op_sequence'] * (1.0 if v['seq_ok'] else 0.0)
    s += w['from_to']     * (1.0 if v['from_to_ok'] else 0.0)
    s += w['legal']       * (1.0 if v['is_legal'] else 0.0)
    s += w['captures']    * (1.0 if v['captures_ok'] else 0.0)
    if v['is_legal']:
        next_board, _ = legal_moves[(v['from_sq'], v['to_sq'])]
        s += w['heuristic'] * heuristic_next_board_value(next_board)
    return s

#  best-of-N sampling 

def _call_vllm(image_arr, camera_name, n, temperature, seed=None):
    """Single chat.completions call that asks vLLM to return `n` independent samples."""
    kw = dict(
        model=CFG.served_name,
        temperature=temperature, top_p=CFG.top_p,
        max_tokens=CFG.max_new_tokens,
        n=n,
        messages=[
            {'role': 'system', 'content': system_prompt_for(camera_name)},
            {'role': 'user', 'content': [
                {'type': 'text',      'text': 'Current board state shown below. Pick the best red move and emit the action sequence.'},
                {'type': 'image_url', 'image_url': {'url': image_to_data_url(image_arr)}},
            ]},
        ],
    )
    if seed is not None:
        kw['extra_body'] = {'seed': seed}
    resp = client.chat.completions.create(**kw)
    return [c.message.content for c in resp.choices]

def ask_llm_best_of_n(image_arr, camera_name, legal_moves, ply_idx=0):
    """Best-of-N generation. Filters hard-invalid candidates, scores the rest, returns the best.
    If every candidate fails the hard gates we retry up to CFG.max_retries more rounds before
    falling back to the highest-scoring invalid candidate (which the game loop will then
    silently remap to the closest legal move).
    """
    history = []            # bookkeeping for dataset trace
    best = None
    for attempt in range(CFG.max_retries + 1):
        texts = _call_vllm(
            image_arr, camera_name,
            n=CFG.candidates_per_turn,
            temperature=CFG.sampling_temperature,
            seed=(CFG.seed + 1000*attempt + ply_idx),
        )
        round_cands = []
        for i, t in enumerate(texts):
            parsed = parse_llm_output(t)
            # Hard gates.
            if CFG.hard_gate_structure and not (parsed['has_thinking'] and parsed['has_actions']):
                round_cands.append({'raw': t, 'parsed': parsed, 'valid': None,
                                    'score': -1.0, 'hard_fail': 'structure'})
                continue
            if CFG.hard_gate_parseable and parsed['parse_error'] is not None:
                round_cands.append({'raw': t, 'parsed': parsed, 'valid': None,
                                    'score': -1.0, 'hard_fail': 'parseable'})
                continue
            v = validate_candidate(parsed, legal_moves)
            s = score_candidate(v, legal_moves)
            round_cands.append({'raw': t, 'parsed': parsed, 'valid': v, 'score': s, 'hard_fail': None})
        history.append({'attempt': attempt, 'candidates': round_cands})

        # Pick the best candidate of this round that passed the hard gates.
        passed = [c for c in round_cands if c['hard_fail'] is None]
        pool   = passed if passed else round_cands
        round_best = max(pool, key=lambda c: c['score'])
        if best is None or round_best['score'] > best['score']:
            best = round_best
        if best['score'] >= CFG.min_score_accept and best.get('valid') and best['valid']['is_legal']:
            break
    return best, history


## 7. Game loop

Each LLM turn:
1. Render the camera frame and save it as the user-image for that turn.
2. Compute the set of legal moves for `R` from `checkers_core`.
3. Call `ask_llm_best_of_n` to get the best-scoring candidate (after up to `CFG.max_retries` rounds).
4. If the best candidate is fully validated *and* its `(from,to)` is legal, execute it verbatim. Otherwise remap to the nearest legal move and mark the turn `was_legal=False` so downstream filters can drop it.
5. Persist the winning candidate's thinking + actions + full validation breakdown + candidate trace.

In [ ]:
def legal_moves_by_fromto(core_board, pin):
    out = {}
    for nxt in possibleMoves(core_board, pin):
        fr, to, cap = diff_boards(core_board, nxt)
        if fr is None or to is None: continue
        out[(fr, to)] = (nxt, cap)
    return out

def choose_closest_legal(intended, legal_keys):
    if not legal_keys: return None
    if intended in legal_keys: return intended
    fi, ti = intended
    def dist(k):
        f, t = k
        if fi is None or ti is None: return 1e9
        fr1, fc1 = name_to_rc(f);  tr1, tc1 = name_to_rc(t)
        try:
            fr2, fc2 = name_to_rc(fi); tr2, tc2 = name_to_rc(ti)
        except Exception:
            return 1e9
        return (fr1-fr2)**2 + (fc1-fc2)**2 + (tr1-tr2)**2 + (tc1-tc2)**2
    return min(legal_keys, key=dist)

def candidate_history_summary(history):
    """Flatten the validation trace into a small list of per-candidate dicts for the dataset row."""
    out = []
    for round_ in history:
        for c in round_['candidates']:
            out.append({
                'attempt':   round_['attempt'],
                'score':     round(c['score'], 4),
                'hard_fail': c['hard_fail'],
                'issues':    (c['valid']['issues'] if c.get('valid') else []),
                'is_legal':  (c['valid']['is_legal'] if c.get('valid') else False),
                'from':      (c['valid']['from_sq'] if c.get('valid') else None),
                'to':        (c['valid']['to_sq']   if c.get('valid') else None),
            })
    return out

def play_one_game(game_idx, camera_name):
    reset_scene()
    core_board = initializeBoard()
    turns = []
    outcome = 'draw'
    opp_history = []
    plies = 0

    for ply in range(CFG.max_plies):
        plies = ply + 1
        img = render(camera_name)
        img_path = CFG.images_dir / f'game{game_idx:03d}_ply{ply:03d}_red.png'
        Image.fromarray(img).save(img_path)

        legal = legal_moves_by_fromto(core_board, 'R')
        if not legal:
            outcome = 'llm_loss'; break

        #  Best-of-N sampling with validation 
        best, history = ask_llm_best_of_n(img, camera_name, legal, ply_idx=ply)
        parsed = best['parsed']
        intended = extract_from_to(parsed['actions']) if parsed['actions'] else (None, None)

        # If the best candidate is fully valid, use its exact (from, to); otherwise fall back to
        # the closest legal move so physics and the game state stay consistent.
        if best.get('valid') and best['valid']['is_legal']:
            chosen = (best['valid']['from_sq'], best['valid']['to_sq'])
            valid_flag = True
        else:
            chosen = choose_closest_legal(intended, list(legal.keys()))
            valid_flag = False
        nxt_board, captured = legal[chosen]

        execute_robot_move(chosen[0], chosen[1], color='red', captured_sqs=captured)
        core_board = nxt_board

        turns.append({
            'camera':           camera_name,
            'image_path':       str(img_path.relative_to(CFG.output_dir)),
            'system':           system_prompt_for(camera_name),
            'llm_raw':          best['raw'],
            'thinking':         parsed['thinking'],
            'actions':          parsed['actions'],
            'actions_raw':      parsed['actions_raw'],
            'intended':         {'from': intended[0], 'to': intended[1]},
            'executed':         {'from': chosen[0],   'to': chosen[1], 'captured': list(captured)},
            'was_legal':        valid_flag,
            'validation':       best.get('valid'),
            'score':            best['score'],
            'num_candidates':   sum(len(r['candidates']) for r in history),
            'num_rounds':       len(history),
            'candidate_trace':  candidate_history_summary(history),
        })

        if isGameOver(core_board, 'R'): outcome = 'llm_win'; break

        policy, weights = select_opponent_policy(ply, CFG.max_plies)
        nxt = rl_pick_move(core_board, 'B', 'R', policy)
        if nxt is None:
            outcome = 'llm_win'; break
        fr, to, cap = diff_boards(core_board, nxt)
        if fr is not None and to is not None:
            execute_robot_move(fr, to, color='black', captured_sqs=cap)
        core_board = nxt
        opp_history.append({'policy': policy, 'weights': weights, 'from': fr, 'to': to, 'captured': list(cap)})

        if isGameOver(core_board, 'B'): outcome = 'llm_loss'; break

    return {'outcome': outcome, 'turns': turns, 'opponent': opp_history, 'plies': plies, 'camera': camera_name}

## 8. Drive the games and emit windowed SFT samples

Each game yields `CFG.windows_per_game` samples of length `[window_min..window_max]`. Windows that end on the game's final ply are marked `terminal=True` with the real `outcome` (`llm_win` or `llm_loss`) so the downstream student learns to play until termination.

In [ ]:
random.seed(CFG.seed)

# Verify the server is still alive before running 30 games.
if server.poll() is not None:
    raise RuntimeError(f'vLLM server is not running (exit code {server.returncode}). '
                       f'Re-run cell-8. Last 30 lines of log:\n' +
                       '\n'.join(open(log_path).readlines()[-30:]))
try:
    client.models.list()
except Exception as e:
    raise RuntimeError(f'vLLM server is unreachable: {e}\n'
                       f'Last 30 lines of log:\n' +
                       '\n'.join(open(log_path).readlines()[-30:]))

def windows_from_game(game_record, game_idx):
    turns = game_record['turns']
    n = len(turns)
    if n == 0: return []
    samples = []
    for w in range(CFG.windows_per_game):
        length = random.randint(CFG.window_min, min(CFG.window_max, n))
        start  = random.randint(0, n - length)
        window = turns[start:start+length]
        ends_game = (start + length == n) and game_record['outcome'] in ('llm_win', 'llm_loss')
        samples.append({
            'sample_id':  f'g{game_idx:03d}_w{w:02d}',
            'system':     window[0]['system'],
            'camera':     game_record['camera'],
            'window_len': length,
            'window_span':(start, start+length),
            'turns':      window,
            'terminal':   ends_game,
            'outcome':    game_record['outcome'] if ends_game else 'ongoing',
        })
    return samples

all_samples = []
all_games   = []
for g in range(CFG.num_games):
    cam = random.choice(CAMERAS)
    try:
        rec = play_one_game(g, cam)
    except Exception as exc:
        print(f'[game {g}] failed: {exc}'); continue
    all_games.append(rec)
    ws = windows_from_game(rec, g)
    all_samples.extend(ws)
    print(f'[game {g:03d}] cam={cam} plies={rec["plies"]} outcome={rec["outcome"]} windows={len(ws)}')

print(f'\nTotal games: {len(all_games)} | total windowed samples: {len(all_samples)}')

[game 000] cam=cam_board_top plies=29 outcome=llm_loss windows=6
[game 001] cam=cam_board_top plies=24 outcome=llm_loss windows=6
[game 002] cam=cam_board_top plies=44 outcome=llm_loss windows=6
[game 003] cam=cam_board_top plies=22 outcome=llm_loss windows=6
[game 004] cam=cam_board_top plies=43 outcome=llm_loss windows=6

Total games: 5 | total windowed samples: 30


## 9. Emit HF / Kaggle-friendly files

`samples.jsonl`, `samples.parquet`, `index.csv` + `images/` — all round-trippable through `datasets.load_dataset(...)` and uploadable to both HuggingFace Hub and Kaggle Datasets.

In [ ]:
import pandas as pd, pyarrow as pa, pyarrow.parquet as pq

def sample_to_messages(sample):
    msgs = [{'role': 'system', 'content': sample['system']}]
    for t in sample['turns']:
        msgs.append({'role': 'user', 'content': [
            {'type': 'image', 'image': t['image_path']},
            {'type': 'text',  'text': 'Current board state shown. Emit exactly one move.'},
        ]})
        msgs.append({'role': 'assistant',
                     'content': f"<thinking>{t['thinking']}</thinking>\n<actions>{t['actions_raw']}</actions>"})
    return msgs

def turn_public_meta(t):
    """The per-turn validation / scoring metadata we keep in the dataset row."""
    return {
        'image_path':     t['image_path'],
        'was_legal':      t['was_legal'],
        'score':          t['score'],
        'num_candidates': t['num_candidates'],
        'num_rounds':     t['num_rounds'],
        'validation':     t.get('validation'),
        'executed':       t['executed'],
        'intended':       t['intended'],
    }

jsonl_path   = CFG.output_dir / 'samples.jsonl'
parquet_path = CFG.output_dir / 'samples.parquet'
csv_path     = CFG.output_dir / 'index.csv'

rows_parquet, rows_csv = [], []
with open(jsonl_path, 'w') as f:
    for s in all_samples:
        msgs = sample_to_messages(s)
        turn_meta = [turn_public_meta(t) for t in s['turns']]
        all_legal = all(t['was_legal'] for t in s['turns'])
        record = {
            'sample_id': s['sample_id'],
            'camera':    s['camera'],
            'window_len':s['window_len'],
            'outcome':   s['outcome'],
            'terminal':  s['terminal'],
            'all_legal': all_legal,
            'images':    [t['image_path'] for t in s['turns']],
            'messages':  msgs,
            'turns_meta':turn_meta,
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        rows_parquet.append({
            'sample_id':  s['sample_id'],
            'camera':     s['camera'],
            'window_len': s['window_len'],
            'outcome':    s['outcome'],
            'terminal':   s['terminal'],
            'all_legal':  all_legal,
            'images':     [t['image_path'] for t in s['turns']],
            'messages':   json.dumps(msgs,      ensure_ascii=False),
            'turns_meta': json.dumps(turn_meta, ensure_ascii=False),
        })
        rows_csv.append({
            'sample_id':    s['sample_id'],
            'camera':       s['camera'],
            'window_len':   s['window_len'],
            'outcome':      s['outcome'],
            'terminal':     s['terminal'],
            'all_legal':    all_legal,
            'avg_score':    round(sum(t['score'] for t in s['turns']) / max(1, len(s['turns'])), 3),
            'n_images':     len(s['turns']),
            'first_image':  s['turns'][0]['image_path'],
        })

pq.write_table(pa.Table.from_pylist(rows_parquet), parquet_path)
pd.DataFrame(rows_csv).to_csv(csv_path, index=False)
print('Wrote:')
print('  ', jsonl_path,   f'({jsonl_path.stat().st_size/1e6:.2f} MB)')
print('  ', parquet_path, f'({parquet_path.stat().st_size/1e6:.2f} MB)')
print('  ', csv_path,     f'({csv_path.stat().st_size/1e3:.1f} KB)')
print('  ', len(list(CFG.images_dir.glob("*.png"))), 'PNG frames in', CFG.images_dir)

Wrote:
   /content/robot_checkers_dataset/samples.jsonl (0.24 MB)
   /content/robot_checkers_dataset/samples.parquet (0.06 MB)
   /content/robot_checkers_dataset/index.csv (2.5 KB)
   162 PNG frames in /content/robot_checkers_dataset/images


## 10. Sanity-load with `datasets`

Confirms the file round-trips through HF Datasets (which is what both `push_to_hub` and Kaggle upload expect).

In [ ]:
from huggingface_hub import HfApi

HF_REPO = "akhil9306/robot-checkers-dataset"

#  Resolve token from platform secret store 
if "HF_TOKEN" not in os.environ:

      try:
          from google.colab import userdata
          os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
          print("HF token loaded from Colab Secrets.")
      except Exception as e:
          raise RuntimeError(
              "Add your HF write token via the key icon in the Colab sidebar (name: HF_TOKEN)"
          ) from e

token = os.environ["HF_TOKEN"]

#  Verify the output directory has data before uploading 
n_samples = sum(1 for _ in open(jsonl_path)) if jsonl_path.exists() else 0
n_images  = len(list(CFG.images_dir.glob("*.png")))
if n_samples == 0:
    raise RuntimeError("samples.jsonl is empty — re-run the game loop cell (cell 8) first.")
print(f"Uploading {n_samples} samples and {n_images} images to {HF_REPO} ...")

#  Create the repo (idempotent) 
api = HfApi(token=token)
api.create_repo(
    repo_id=HF_REPO,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

#  Upload the entire output directory 
# This includes: samples.jsonl, samples.parquet, index.csv, images/*.png
api.upload_folder(
    folder_path=str(CFG.output_dir),
    repo_id=HF_REPO,
    repo_type="dataset",
    commit_message=f"Add {n_samples} windowed SFT samples ({n_images} images)",
)

print(f"\nDone! Dataset live at: https://huggingface.co/datasets/{HF_REPO}")

HF token loaded from Colab Secrets.
Uploading 30 samples and 162 images to akhil9306/robot-checkers-dataset ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...es/game000_ply000_red.png: 100%|##########| 90.4kB / 90.4kB            

  ...es/game000_ply001_red.png:   4%|4         | 3.64kB / 84.2kB            

...

  ...es/game000_ply006_red.png:   4%|4         | 3.52kB / 81.4kB            

  ...es/game000_ply007_red.png:   4%|4         | 3.47kB / 80.1kB            

  ...es/game000_ply008_red.png:   4%|4         | 3.44kB / 79.6kB            

  ...es/game000_ply009_red.png:   4%|4         | 3.47kB / 80.1kB            


Done! Dataset live at: https://huggingface.co/datasets/akhil9306/robot-checkers-dataset


In [ ]:
import zipfile
from pathlib import Path

def zip_folder(src: Path, dst: Path):
    """Zip an entire folder tree into dst. Paths inside the zip are relative to src.parent."""
    if not src.exists():
        print(f"  Skipping {src.name} — folder not found at {src}")
        return
    total = 0
    with zipfile.ZipFile(dst, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for f in sorted(src.rglob('*')):
            if f.is_file():
                zf.write(f, f.relative_to(src.parent))
                total += 1
    size_mb = dst.stat().st_size / 1e6
    print(f"  {dst.name}  →  {total} files  {size_mb:.1f} MB  ({dst})")

work = Path('/kaggle/working') if _IS_KAGGLE else Path('/content')

folders = {
    'robot_checkers_dataset': work / 'robot_checkers_dataset',
    'sample_data':            work / 'sample_data',
}

print("Zipping folders...")
for name, src in folders.items():
    zip_folder(src, work / f'{name}.zip')

print("\nDone. Download from the Output tab on Kaggle (or the Files panel on Colab).")

Zipping folders...
  robot_checkers_dataset.zip  →  166 files  12.1 MB  (/content/robot_checkers_dataset.zip)
  sample_data.zip  →  6 files  7.1 MB  (/content/sample_data.zip)

Done. Download from the Output tab on Kaggle (or the Files panel on Colab).


## 11. Shut down the server

In [ ]:
try:
    server.terminate(); server.wait(timeout=30)
except Exception:
    server.kill()
log_fh.close()
print('vLLM shut down.')

vLLM shut down.
